# Normalized gain: the three facts

1. The 12 top-row curves of Figure 5 collapse to **one curve** (minorized) and **one family in $k$** (exact).
2. The $k$ ranges quoted in the email (0.9–6.9 and 0.1–0.7) correspond to $\sqrt{\tilde v}\,\|x\|/b_\eta$.
3. The exact gain starts below the minorized one and **overtakes it for every $k$**; a smaller $k$ moves the crossover to larger errors.
4. In Ignacio's realistic scenario, most steps fall **below** that crossover, where the exact gain is smaller.

Gain functions and operating point are copied from `gain-functions.ipynb` (§2 and the §1 printout).
That notebook validates the exact gain (63) against the numerical inversion of eq. (18).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.special import log_ndtr, logsumexp

%config InlineBackend.figure_format = 'svg'
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

# operating point, from gain-functions.ipynb §1
X, b_eta, v_tilde = 1.573030, 0.158114, 0.043831
X_PCT = [0.6735, 1.5713, 2.5910]          # 10/50/90 % percentiles of ||x_t||
S = np.array([1.0, -1.0])                  # the two branches, +1 and -1


def g1_min(e, b, v, X):                    # eq. (50)
    return v * e / (b * np.abs(e) + v * X**2)


def g1_exact(e, b, v, X):                  # eq. (63)
    e = np.asarray(e, dtype=float)[..., None]
    kappa = (S * e - v * X**2 / b) / (np.sqrt(v) * X)
    log_pi = -S * e / b + log_ndtr(kappa)
    pi = np.exp(log_pi - logsumexp(log_pi, axis=-1, keepdims=True))
    mills = np.exp(-0.5 * kappa**2 - 0.5 * np.log(2 * np.pi) - log_ndtr(kappa))
    Lam, Gam = np.sum(S * pi, axis=-1), np.sum(S * pi * mills, axis=-1)
    return v * Lam / b - np.sqrt(v) * Gam / X

## 1. The collapse

Define
$$\chi = g_1\|x\|^2 \;(\text{change of the output}),\qquad \tau = \frac{\tilde v\|x\|^2}{b_\eta},\qquad u = \frac{e}{\tau},\qquad k = \frac{\tilde v\|x\|^2}{b_\eta^2}.$$

**Minorized.** Multiply (50) by $\|x\|^2$, divide top and bottom by $b_\eta$, then by $\tau$:
$$\chi = \frac{\tau e}{|e|+\tau} \;\Rightarrow\; \frac{\chi}{\tau} = \frac{u}{|u|+1}.$$
No parameter left, so every $(\tilde v, b_\eta, \|x\|)$ gives the same curve.

**Exact.** Substituting $e = u\tau$ in (63) gives $e/b_\eta = ku$ and $\sqrt{\tilde v}\|x\|/b_\eta = \sqrt k$:
$$\kappa_\pm = \sqrt k(\pm u - 1),\qquad \log\pi_\pm = \mp k u + \log\Phi(\kappa_\pm),\qquad \frac{\chi}{\tau} = \Lambda - \frac{\Gamma}{\sqrt k}.$$
One parameter left, $k$, so the curves form a family indexed by $k$.

**Test.** Take the 12 parameter sets of Figure 5, plot them raw and normalized. The code uses only
the original `g1_min` / `g1_exact`; the normalized formulas above are not used, so the plot checks them.

In [ ]:
FIG5 = ([dict(v=v, b=b_eta, X=X) for v in (0.005, v_tilde, 0.2, 2.0)]
        + [dict(v=v_tilde, b=b, X=X) for b in (np.sqrt(1e-3), b_eta, 5 * b_eta, 20 * b_eta)]
        + [dict(v=v_tilde, b=b_eta, X=x) for x in X_PCT + [2 * X_PCT[2]]])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
e = np.linspace(-40, 40, 1601) * b_eta
cmap, lognorm = plt.get_cmap("viridis"), plt.Normalize(-2, 2.3)
worst = 0.0
for p in FIG5:
    k, tau = p["v"] * p["X"]**2 / p["b"]**2, p["v"] * p["X"]**2 / p["b"]
    c = cmap(lognorm(np.log10(k)))
    axL.plot(e, g1_exact(e, **p), color=c); axL.plot(e, g1_min(e, **p), color=c, ls="--", lw=0.8)
    u = np.linspace(-6, 6, 601)            # normalized error, mapped back to raw e = u * tau
    axR.plot(u, g1_exact(u * tau, **p) * p["X"]**2 / tau, color=c)
    worst = max(worst, np.max(np.abs(g1_min(u * tau, **p) * p["X"]**2 / tau - u / (np.abs(u) + 1))))
axR.plot(u, u / (np.abs(u) + 1), "k--", lw=1.5, label=r"minorized, all 12: $u/(|u|+1)$")
axL.set(xlabel=r"$e_t$", ylabel=r"$g_1$", title="Figure 5 top row, raw (solid exact, dashed minorized)")
axR.set(xlabel=r"$u = e_t/\tau$", ylabel=r"$\chi/\tau$", title=r"same 12 sets, normalized (exact, colour = $\log_{10}k$)")
axR.legend(fontsize=8)
for ax in (axL, axR): ax.grid(alpha=0.3)
fig.colorbar(plt.cm.ScalarMappable(norm=lognorm, cmap=cmap), ax=axR, label=r"$\log_{10} k$")
plt.show()
print(f"minorized, 12 sets: max |chi/tau - u/(|u|+1)| = {worst:.1e}")

# same k, very different parameters -> same exact curve
p1 = dict(v=0.001, b=0.05, X=0.5)
p2 = dict(v=5.0, b=np.sqrt(5.0 * 3.0**2 / (0.001 * 0.5**2 / 0.05**2)), X=3.0)   # b chosen so k2 = k1
curve = lambda p: g1_exact(u * p["v"] * p["X"]**2 / p["b"], **p) * p["b"] / p["v"]
print(f"exact, k = {0.001*0.5**2/0.05**2:g} from two sets 5000x apart in v~: max difference = {np.max(np.abs(curve(p1) - curve(p2))):.1e}")

## 2. Which parameter: $k$ or $\sqrt k$

The email gives $k$ from 0.9 to 6.9 for the visible curves and 0.1 to 0.7 for the small ones.
With $k = \tilde v\|x\|^2/b_\eta^2$, the Figure 5 values span 0.01 to 198, so the quoted ranges don't fit.
With $\sqrt k = \sqrt{\tilde v}\|x\|/b_\eta$ they do: the $\|x\|$ sweep gives 0.89 to 6.86,
and the three smallest values are 0.10, 0.42, 0.70.

$\sqrt k$ is the prediction's standard deviation $\sqrt{\tilde v}\|x\|$ over $b_\eta$, and it is also the
factor in $\kappa_\pm = \sqrt k(\pm u - 1)$. To confirm: is this the intended definition?

In [ ]:
names = ["v~"] * 4 + ["b_eta"] * 4 + ["||x||"] * 4
print(f"{'sweep':>6s} {'v~':>8s} {'b_eta':>8s} {'||x||':>7s} {'k':>9s} {'sqrt(k)':>8s}")
for nm, p in zip(names, FIG5):
    k = p["v"] * p["X"]**2 / p["b"]**2
    print(f"{nm:>6s} {p['v']:8.4g} {p['b']:8.4g} {p['X']:7.4f} {k:9.4g} {np.sqrt(k):8.3f}")

## 3. Where the exact gain overtakes the minorized one

For each $k$: the $u$ where exact − minorized changes sign, and the $u$ where each reaches 99 %
of its saturation. $u>0$ is enough since both gains are odd.

**Left:** the difference. Every curve is negative first (minorized larger) and positive after its dot $u^*$ (exact larger).
**Right:** the gains themselves for the two smallest $k$. The minorized gain is larger up to $u^*$ (4.9 for $k=0.1$,
20.6 for $k=0.01$); after that the exact gain saturates at 1 while the minorized one is still climbing.

In [ ]:
def chi_tau_exact(u, k):                   # eq. (63) in normalized form (section 1)
    return g1_exact(u * k, 1.0, k, 1.0) / k    # b = 1, X = 1, v = k  gives tau = k


u = np.logspace(-3, 5, 20001)
mino = u / (u + 1)
print(f"{'k':>6s} {'sqrt(k)':>8s} {'crossover u*':>13s} {'u(99%) exact':>13s} {'u(99%) minorized':>17s}")
fig, (ax, axG) = plt.subplots(1, 2, figsize=(13, 4.3), constrained_layout=True)
for i, k in enumerate([0.01, 0.1, 0.5, 1, 3, 10, 100]):
    ex = chi_tau_exact(u, k)
    cross = u[np.argmax(ex > mino)]
    print(f"{k:6g} {np.sqrt(k):8.3f} {cross:13.3g} {u[np.argmax(ex >= 0.99)]:13.3g} {u[np.argmax(mino >= 0.99)]:17.3g}")
    ax.semilogx(u, ex - mino, color=COLORS[i], label=f"k = {k:g}")
    ax.plot(cross, 0, "o", color=COLORS[i], ms=5)            # the crossover u*
    if k in (0.01, 0.1):                                     # right panel: the gains themselves
        axG.semilogx(u, ex, color=COLORS[i], label=f"exact, k = {k:g}")
        axG.plot(cross, np.interp(cross, u, ex), "o", color=COLORS[i], ms=6)
axG.semilogx(u, mino, "k--", label="minorized")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel=r"$u = e_t/\tau$", ylabel="exact − minorized", xlim=(1e-2, 1e3),
       title=r"$\chi/\tau$: exact minus minorized  (> 0: exact above; dots: $u^*$)")
axG.set(xlabel=r"$u = e_t/\tau$", ylabel=r"$\chi/\tau$", xlim=(1e-2, 1e3),
        title=r"small $k$: the gains themselves (dots: $u^*$)")
for a in (ax, axG):
    a.grid(alpha=0.3, which="both"); a.legend(fontsize=8)
plt.show()

## 4. Where Ignacio's filters operate

`03_escenario_realista.ipynb` compares the two Laplacian filters in a realistic scenario: $M = 128$, AR(1) input
with unit variance, generalized Gaussian noise ($\beta^* = 0.2$). At equal floor, the exact filter needs more
steps than the minorized one, and the gap shrinks with SNR:

| SNR | 0 dB | 5 dB | 10 dB | 15 dB |
|---|---|---|---|---|
| steps, exact / minorized | 1.63 | 1.34 | 1.12 | 1.03 |

**Question.** Where on the curves of section 3 do those filters actually run?

**How.** That notebook fixes $b_\eta$ (from the SNR) and $\varepsilon$ (from its search), but $\tilde v_t$ is filter
state, so it is not printed anywhere. The cell below reruns the same scenario, with the same $\varepsilon$ values,
and records $\tilde v_t$, $\|x_t\|$ and $e_t$ at every step. From those it gets $\sqrt k$ and $|u|$ at each step,
and counts how many steps fall below the crossover $u^*$, where the exact gain is **smaller** than the minorized one.

Two windows: the **transient** (first 3000 adaptive steps, where convergence speed is decided) and the
**steady state** (second half of the run). One realisation, $N = 48000$. Takes about 30 s.

In [ ]:
from scipy.signal import lfilter
from scipy.special import gammaln
from scipy.stats import gennorm
import rir_generator as rir

# Scenario of 03_escenario_realista.ipynb, section 2
M_R, N_R, WARMUP, AR_A, BETA, VAR0 = 128, 48000, 500, -0.9, 0.2, 2.0
ho = rir.generate(c=340, fs=8000, r=[1, 1.5, 1], s=[1, 2.5, 2], L=[5, 10, 6],
                  reverberation_time=0.2, nsample=M_R).flatten()
ho /= np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M_R), np.arange(M_R)))
P_signal = ho @ (AR_A**lags) @ ho

# epsilon selected by its search (table under its Figure 5): SNR -> (minorized, exact)
EPS = {0: (6.96e-06, 2.28e-06), 5: (7.64e-06, 3.45e-06), 10: (9.15e-06, 4.54e-06), 15: (1.53e-05, 6.39e-06)}
SPEED_RATIO = {0: 1.63, 5: 1.34, 10: 1.12, 15: 1.03}   # its Figure 4


def signals(scale_gg, seed=0):
    rng = np.random.default_rng(seed)
    drive = np.sqrt(1 - AR_A**2) * rng.standard_normal(N_R + WARMUP)
    x = lfilter([1.0], [1.0, -AR_A], drive)[WARMUP:]
    return x, np.convolve(ho, x)[:N_R] + gennorm.rvs(BETA, scale=scale_gg, size=N_R, random_state=rng)


def run(x, d, b, eps, exact):
    # One run of the filter; returns (v~, ||x||, e) at every adaptive step.
    #
    # One row per adaptive step, so the result is (N_R - M_R, 3) = (47872, 3). The three
    # values are the a priori ones, recorded before the update: they are exactly the
    # arguments the gain function is evaluated at on that step, so every row is a genuine
    # evaluation point of the curves of section 3.
    w, v, xt, rec = np.zeros(M_R), VAR0, np.zeros(M_R), []
    for t in range(N_R):
        xt = np.roll(xt, 1); xt[0] = x[t]
        e_t = d[t] - xt @ w                                 # a priori: w not updated yet
        if t < M_R:
            continue                                        # filter not filled: not recorded
        vt = v + eps                                        # the variance THIS step uses
        p = xt @ xt; nx = np.sqrt(p)
        rec.append((vt, nx, e_t))                           # the three numbers the gain sees
        if not exact:                                       # eqs. (50)-(51)
            s = b * abs(e_t) + vt * p
            w = w + xt * (vt * e_t / s)
            v = vt * (1 - vt * p / (M_R * s))
        else:                                               # eqs. (63)-(64)
            kap = (S * e_t - vt * p / b) / (np.sqrt(vt) * nx)
            lp = -S * e_t / b + log_ndtr(kap)
            pi = np.exp(lp - logsumexp(lp))
            h = np.exp(-0.5 * kap**2 - 0.5 * np.log(2 * np.pi) - log_ndtr(kap))
            Lam, Gam, P, Q = np.sum(S * pi), np.sum(S * pi * h), np.sum(pi * h), np.sum(pi * kap * h)
            w = w + (vt * Lam / b - np.sqrt(vt) * Gam / nx) * xt
            g, l = vt / b, np.sqrt(vt) / nx
            v = vt + (g**2 * (1 - Lam**2) - 2 * g * l * (P - Lam * Gam) - l**2 * (Q + Gam**2)) * p / M_R
    return np.array(rec)


u_fine = np.logspace(-4, 8, 40001)
ops = []
for snr, (eps_min, eps_ex) in EPS.items():
    var_eta = P_signal / 10**(snr / 10)
    b = np.sqrt(var_eta / 2)
    # signals() defaults to seed=0, so every number below comes from ONE realisation.
    x, d = signals(np.sqrt(var_eta / np.exp(gammaln(3 / BETA) - gammaln(1 / BETA))))
    for name, eps, exact in (("minorized", eps_min, False), ("exact", eps_ex, True)):
        rec = run(x, d, b, eps, exact)
        # The windows slice rec, not t. rec[0] is t = M_R, so "steady" starts at t = 24128
        # and not at 24000: an off-by-M_R, harmless here. transient = 3000 steps
        # (t = 128..3127), steady = 23872 steps (the second half of the run).
        for window, r in (("transient", rec[:3000]), ("steady", rec[N_R // 2:])):
            vt, nx, e_t = r.T
            sk = np.sqrt(vt) * nx / b                       # sqrt(k) at each step
            uu = np.abs(e_t) * b / (vt * nx**2)             # |u| at each step
            # sk and uu hold one value per step: the filter does not sit at an operating
            # point, it traces a cloud. What follows collapses that cloud to plain order
            # statistics, unweighted -- one vote per time step, no weighting by the size of
            # the error or by how much the step moved w.
            #
            # sk is Leszek's k; sections 1-4 call it sqrt(k) because their k is v~||x||^2/b^2.
            # The median is invariant under that reparametrisation: sk >= 0 and squaring is
            # monotone there, so median(sk)**2 == median(sk**2). The round trip below (**2
            # here, sqrt in sk_med) is exact and "median k" means the same in both
            # conventions. A mean would NOT have that property.
            k_med = np.median(sk)**2
            # u_star, and the curve plotted in the next cell, are evaluated at k_med: they
            # are curve(median k), not median(curve(k)). chi_tau_exact is nonlinear in k, so
            # the two differ. Steady state: sk spans 0.41-0.73 (5-95%) at 0 dB, so the gap is
            # small. Transient: sk spans 0.44-11.38, where one curve is not representative --
            # which is why the figure of the next cell shows the steady state only.
            u_star = u_fine[np.argmax(chi_tau_exact(u_fine, k_med) > u_fine / (u_fine + 1))]
            # sk_med == np.median(sk) exactly (sqrt of the square above). np.percentile(.,50)
            # and np.median agree: both average the two middle values on an even-length array.
            # k barely moves in steady state while |u| spans a decade, so |u| is the one that
            # needs a spread (u90) rather than a single number. u50 and k_med are taken
            # independently, so the pair (u50, k_med) need not be a step that actually occurred.
            ops.append(dict(snr=snr, filt=name, win=window, b=b, v=np.median(vt), nx=np.median(nx),
                            sk_lo=np.percentile(sk, 5), sk_med=np.sqrt(k_med), sk_hi=np.percentile(sk, 95),
                            u50=np.percentile(uu, 50), u90=np.percentile(uu, 90), u_star=u_star,
                            below=100 * np.mean(uu < u_star)))

for window in ("transient", "steady"):
    print(f"--- {window} ---")
    print(f"{'SNR':>4} {'filter':>9} {'b_eta':>6} {'v~':>8} {'||x||':>6} {'sqrt k (5-50-95%)':>19} "
          f"{'|u| 50%':>7} {'|u| 90%':>7} {'u*':>6} {'% steps below u*':>17}")
    for o in ops:
        if o["win"] == window:
            print(f"{o['snr']:>4} {o['filt']:>9} {o['b']:6.3f} {o['v']:8.2e} {o['nx']:6.2f} "
                  f"{o['sk_lo']:5.2f} - {o['sk_med']:4.2f} - {o['sk_hi']:5.2f} "
                  f"{o['u50']:7.2f} {o['u90']:7.2f} {o['u_star']:6.2f} {o['below']:17.0f}")

**Figure.** For the exact filter at each SNR (steady state): exact − minorized gain at the median $k$.
The dot is the median step's $|u|$; the dotted vertical line is where 90 % of steps lie to its left.
Below zero, the exact filter takes a smaller step than the minorized one.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.3), constrained_layout=True)
u_plot = np.logspace(-3, 2, 1500)
exact_steady = [o for o in ops if o["filt"] == "exact" and o["win"] == "steady"]
for i, o in enumerate(exact_steady):
    diff = chi_tau_exact(u_plot, o["sk_med"]**2) - u_plot / (u_plot + 1)
    ax.semilogx(u_plot, diff, color=COLORS[i],
                label=f"SNR {o['snr']} dB:  sqrt k = {o['sk_med']:.2f},  steps exact/minorized = {SPEED_RATIO[o['snr']]}")
    ax.plot(o["u50"], np.interp(o["u50"], u_plot, diff), "o", color=COLORS[i], ms=7)
    ax.axvline(o["u90"], color=COLORS[i], ls=":", lw=1)
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel=r"$|u| = |e_t|/\tau$", ylabel=r"$\chi/\tau$: exact − minorized",
       title="Ignacio's operating points (exact filter, steady state)")
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8, loc="lower left")
plt.show()

### Findings

* **Operating range.** $\|x\| \approx 11$ (that is $\sqrt{128}$) and $\tilde v \approx 10^{-3}$ at every SNR, so $\sqrt k$ is set
  mainly by $b_\eta$: about 0.4–0.8 at 0 dB, up to 2–6 at 15 dB. This is the range quoted in the email.
* **Most steps are below the crossover.** In both windows and for both filters, the large majority of steps
  (the last column) have $|u| < u^*$, the region where the exact gain is smaller than the minorized one.
  A smaller gain on most steps is a direct reason for the exact filter to converge more slowly.
* **The trend with SNR matches.** At the median step of the exact filter (steady state), its gain is this
  fraction of the minorized gain, computed from the section 3 curves at the median $k$ and median $|u|$:

  | SNR | 0 dB | 5 dB | 10 dB | 15 dB |
  |---|---|---|---|---|
  | exact gain / minorized gain, median step | 0.50 | 0.61 | 0.75 | 0.88 |
  | steps, exact / minorized (Ignacio) | 1.63 | 1.34 | 1.12 | 1.03 |

  The smaller the exact gain relative to the minorized one, the larger the step-count gap. Higher SNR means
  larger $k$, where the two gains are closer near zero error.

**What this does not show.** It is consistent with the speed result, not a proof of its cause:
* the two filters run with different $\varepsilon$ and settle at different $\tilde v$;
* the variance recursion (the bottom row of Figure 5) is not separated out here;
* one realisation only.

## Summary

* **Collapse:** with $\chi = g_1\|x\|^2$ and $\tau = \tilde v\|x\|^2/b_\eta$, the minorized gain is the single
  curve $u/(|u|+1)$ and the exact gain depends only on $k$. Both checks in section 1 hold to roundoff.
* **Parameter:** the quoted ranges match $\sqrt{\tilde v}\,\|x\|/b_\eta$ (section 2), to be confirmed.
* **Ordering:** for every $k$, the exact gain starts below the minorized one, overtakes it at $u^*$, and
  reaches saturation first. $u^*$ grows as $k$ shrinks: 1.0 at $k=1$, 4.9 at $k=0.1$, 20.6 at $k=0.01$
  (section 3). For small $k$ the crossover lies beyond the errors that dominate Figure 5, which is why
  the exact gain appears below the minorized one there.
* **Ignacio's scenario:** $\sqrt k \approx$ 0.4 to 6 across SNR, and most steps sit below the crossover, where the
  exact gain is smaller. That is consistent with the exact filter being slower, and with the gap shrinking as SNR
  (and $k$) grows (section 4). Consistent, not proven.

## 5. Corrections after Leszek's reply

Leszek defines $k = \sqrt{\tilde v}\,\|x\|/b_\eta$. Sections 1 to 4 use the name $k$ for $\tilde v\|x\|^2/b_\eta^2$,
which is **his $k^2$**; they are left as they were. From here on, $k$ is his. In this $k$ the exact gain reads
$$\kappa_\pm = k(\pm u - 1),\qquad \log\pi_\pm = \mp k^2 u + \log\Phi(\kappa_\pm),\qquad \frac{\chi}{\tau} = \Lambda - \frac{\Gamma}{k}.$$

* **Crossover values.** The $u^*$ of section 3 are right for the old $k$, but the email put them next to ranges
  in this $k$ ("4.9 at $k = 0.1$" should read "4.9 at $k \approx 0.32$"). The first table below gives $u^*$ in this $k$.
* **Ignacio's range.** Section 4 already prints this $k$ (column `sqrt k`), so "about 0.4 to 6" stands.
* **Common $\tau$.** In section 4 the fraction of steps below $u^*$ uses each filter's own $\tilde v_t$, so its own
  $\tau$ and $k$. The gain ratio in its findings (0.50 to 0.88) does not: it evaluates both gains at the exact
  filter's median $k$ and $|u|$, so the minorized gain borrows the exact filter's $\tau$. At equal floor the two
  filters settle at different $\tau$, so the second table compares the effective step $\chi/e$, each filter at its
  own median step: $1/(1+|u|)$ for the minorized filter, $(\chi/\tau)/|u|$ for the exact one.

In [ ]:
# Crossover u* in Leszek's k. chi_tau_exact takes the old k, so it gets k**2.
print(f"{'k':>6s} {'u*':>8s}")
for k in [0.01, 0.1, 0.2, 0.316, 0.4, 0.63, 1, 2.45, 3, 6]:
    print(f"{k:6g} {u_fine[np.argmax(chi_tau_exact(u_fine, k**2) > u_fine / (u_fine + 1))]:8.3g}")

# Gain ratio exact / minorized at the median step: common tau (section 4) against each filter's own tau.
op = lambda filt, snr, win: next(o for o in ops if o["filt"] == filt and o["snr"] == snr and o["win"] == win)
tau = lambda o: o["v"] * o["nx"]**2 / o["b"]            # tau at the median v~ and ||x||
for window in ("transient", "steady"):
    print(f"\n--- {window} ---")
    print(f"{'SNR':>4} {'tau min/exact':>13} {'common tau':>11} {'own tau':>8} {'steps, exact/min':>17}")
    for snr in EPS:
        mi, ex = (op(f, snr, window) for f in ("minorized", "exact"))
        chi_tau = float(chi_tau_exact(ex["u50"], ex["sk_med"]**2))
        common = chi_tau / (ex["u50"] / (ex["u50"] + 1))     # both gains at the exact filter's u and k
        own = chi_tau / ex["u50"] * (1 + mi["u50"])          # chi/e, each filter at its own median step
        print(f"{snr:>4} {tau(mi) / tau(ex):13.2f} {common:11.2f} {own:8.2f} {SPEED_RATIO[snr]:17.2f}")

### Findings

* **$u^*$ in Leszek's $k$:** 294, 20.6, 8.8, 3.6, 1.02 and 0.18 at $k$ = 0.01, 0.1, 0.2, 0.4, 1 and 3, his values.
  Over Ignacio's steady-state range, $k \approx 0.4$ to 6, $u^*$ runs from about 3.6 down to 0.05.
* **Each filter's own $\tau$ changes the gain ratio only a little.** At equal floor the minorized $\tau$ is 1.3 to
  2.1 times the exact one at steady state (Leszek's fixed-variance analysis gives 1.2 to 2). With each filter at its
  own median step, the exact step is 0.46, 0.59, 0.73 and 0.87 of the minorized one at 0, 5, 10 and 15 dB, against
  0.50 to 0.88 with the common $\tau$; 0.60 to 0.92 in the transient. The larger $\tau$ lowers the minorized filter's
  $|u|$, which enlarges its step, so the ratio drops by a few percent. The trend with SNR is unchanged.
* Still consistent with the speed gap, not a proof: the speed depends on the whole error distribution, not on one step.

## 6. Does the variance recursion explain the gap?

Leszek's test, changing one thing at a time. Section 6 of the draft predicts exact / minorized step ratios of
1.13, 1.05, 1.00 and 0.99 at 0, 5, 10 and 15 dB for **fixed-variance** filters (white input, $M = 16$, Gaussian
treatment of the a priori error). Ignacio's sKF-L runs give 1.63, 1.34, 1.12 and 1.03.

**The sKF rows** rerun the SNR sweep of `03_escenario_realista.ipynb`: same room, AR(1) input, generalized Gaussian
noise and seeds (the signals are generated at its length, 96000, so the noise samples are identical), $M = 128$,
$w_0 = 0$, target $-20$ dB, $\varepsilon$ on 11 grid values over 5 decades, search on $R = 3$, check on $R = 10$,
floor = mean of the last quarter, converged = first step within 3 dB of the floor.

**The fKF rows** change only the variance: $\tilde v_t$ is frozen at a constant $\tilde v$, and that constant is
tuned in place of $\varepsilon$. The gains are unchanged, eqs. (50) and (63).

**Two departures from notebook 03.**
* Its 5 dB ratio came from its long run ($N = 96000$, $R = 20$). Here every SNR uses the same $N$ and $R$.
* Everything runs at $N = 24000$ (its search length) and again at $N = 48000$: at 0 dB the exact filter needs
  close to half of 24000 steps, so the longer run checks that the floors had settled.

**Reading.** If the fKF ratio at 0 dB drops toward 1.13, the variance recursion explains the extra gap. If it
stays near 1.6, the cause is something else that differs from the analysis: the input, $M$ (128 against 16), or
the criterion.

The update lines are those of `run` in section 4, vectorised over grid values and realisations. About 2 min.

In [ ]:
N_03 = 96000                                    # notebook 03 generates its signals at this length
R_SEARCH, R_CHECK, TARGET_DB = 3, 10, -20.0
GRID = {"sKF": np.logspace(-8, -3, 11),         # epsilon, as in notebook 03
        "fKF": np.logspace(-6, -1, 11)}         # the frozen v~
PREDICTED = {0: 1.13, 5: 1.05, 10: 1.00, 15: 0.99}   # section 6 of the draft


def signals_03(snr, seed, n):
    # generate_signals of notebook 03 at this SNR, cut to its first n samples
    var_eta = P_signal / 10**(snr / 10)
    rng = np.random.default_rng(seed)
    drive = np.sqrt(1 - AR_A**2) * rng.standard_normal(N_03 + WARMUP)
    x = lfilter([1.0], [1.0, -AR_A], drive)[WARMUP:]
    scale = np.sqrt(var_eta / np.exp(gammaln(3 / BETA) - gammaln(1 / BETA)))
    d = np.convolve(ho, x)[:N_03] + gennorm.rvs(BETA, scale=scale, size=N_03, random_state=rng)
    return x[:n], d[:n]


def misalignment_batch(X, D, b, par, exact, frozen):
    # B filters side by side, one per row of X, D. par: epsilon (sKF) or the frozen v~ (fKF).
    # Returns the misalignment before each update, shape (B, n); ||ho|| = 1.
    B, n = X.shape
    w, xt, v = np.zeros((B, M_R)), np.zeros((B, M_R)), np.full(B, VAR0)
    mis = np.empty((B, n))
    for t in range(n):
        xt = np.roll(xt, 1, axis=1); xt[:, 0] = X[:, t]
        e = D[:, t] - np.einsum("bm,bm->b", xt, w)
        mis[:, t] = np.einsum("bm,bm->b", w - ho, w - ho)
        if t < M_R:
            continue
        vt = par if frozen else v + par
        p = np.einsum("bm,bm->b", xt, xt)
        if not exact:                                       # eqs. (50)-(51)
            s = b * np.abs(e) + vt * p
            w = w + xt * (vt * e / s)[:, None]
            if not frozen:
                v = vt * (1 - vt * p / (M_R * s))
        else:                                               # eqs. (63)-(64)
            nx = np.sqrt(p)
            kap = (S * e[:, None] - (vt * p / b)[:, None]) / (np.sqrt(vt) * nx)[:, None]
            lp = -S * e[:, None] / b + log_ndtr(kap)
            pi = np.exp(lp - logsumexp(lp, axis=1, keepdims=True))
            h = np.exp(-0.5 * kap**2 - 0.5 * np.log(2 * np.pi) - log_ndtr(kap))
            Lam, Gam = (S * pi).sum(1), (S * pi * h).sum(1)
            w = w + xt * (vt * Lam / b - np.sqrt(vt) * Gam / nx)[:, None]
            if not frozen:
                P, Q = (pi * h).sum(1), (pi * kap * h).sum(1)
                g, l = vt / b, np.sqrt(vt) / nx
                v = vt + (g**2 * (1 - Lam**2) - 2 * g * l * (P - Lam * Gam) - l**2 * (Q + Gam**2)) * p / M_R
    return mis


def steady_state(misalignment):
    # notebook 03: floor = mean of the last quarter, converged = first step within 3 dB of it
    floor = 10 * np.log10(misalignment[3 * len(misalignment) // 4:].mean())
    return floor, int(np.argmax(10 * np.log10(misalignment) < floor + 3))


def pick(grid, floors):
    # pick_epsilon of notebook 03: interpolate on the branch above the best floor (below it the filter stalls)
    best = int(np.argmin(floors))
    g, f = grid[best:], floors[best:]
    if TARGET_DB > f.max():
        return g[-1], "grid too narrow"
    if TARGET_DB < f.min():
        return g[0], "target not reached"
    order = np.argsort(f)
    return 10**np.interp(TARGET_DB, f[order], np.log10(g)[order]), "ok" if best > 0 else "ok (optimum at grid edge)"


def ratio(n, snr, variant):
    # steps exact / steps minorized; nan if either missed the target
    (_, _, s_min, st_min), (_, _, s_ex, st_ex) = (sweep[(n, snr, variant, f)] for f in ("minorized", "exact"))
    return s_ex / s_min if st_min.startswith("ok") and st_ex.startswith("ok") else np.nan

In [ ]:
sweep = {}                                      # (N, SNR, variant, filter) -> (parameter, floor, steps, status)
for n in (24000, 48000):
    for snr in EPS:
        b = np.sqrt(P_signal / 10**(snr / 10) / 2)
        X, D = map(np.array, zip(*[signals_03(snr, seed, n) for seed in range(R_CHECK)]))
        for variant, grid in GRID.items():
            for filt in ("minorized", "exact"):
                exact, frozen = filt == "exact", variant == "fKF"
                Xs, Ds = np.repeat(X[:R_SEARCH], len(grid), axis=0), np.repeat(D[:R_SEARCH], len(grid), axis=0)
                curves = misalignment_batch(Xs, Ds, b, np.tile(grid, R_SEARCH), exact, frozen)
                floors = np.array([steady_state(c)[0] for c in curves.reshape(R_SEARCH, len(grid), n).mean(axis=0)])
                par, status = pick(grid, floors)
                floor, steps = steady_state(misalignment_batch(X, D, b, np.full(R_CHECK, par), exact, frozen).mean(axis=0))
                sweep[(n, snr, variant, filt)] = (par, floor, steps, status)

for n in (24000, 48000):
    print(f"--- N = {n} ---")
    print(f"{'SNR':>4} {'':>4} {'filter':>9} {'eps or v~':>10} {'floor':>7} {'steps':>6}  status")
    for snr in EPS:
        for variant in GRID:
            for filt in ("minorized", "exact"):
                par, floor, steps, status = sweep[(n, snr, variant, filt)]
                print(f"{snr:>4} {variant:>4} {filt:>9} {par:10.2e} {floor:7.2f} {steps:6d}  {status}")

eps_here = np.array([[sweep[(24000, snr, "sKF", f)][0] for f in ("minorized", "exact")] for snr in EPS])
print(f"\nsKF at N = 24000 against notebook 03: largest epsilon difference "
      f"{np.max(np.abs(eps_here / np.array(list(EPS.values())) - 1)):.2%}")

print(f"\nsteps, exact / minorized, at equal floor")
print(f"{'SNR':>4} {'notebook 03':>12} {'sKF N=24000':>12} {'fKF N=24000':>12} {'sKF N=48000':>12} {'fKF N=48000':>12} {'analysis':>9}")
for snr in EPS:
    print(f"{snr:>4} {SPEED_RATIO[snr]:12.2f} "
          + " ".join(f"{ratio(n, snr, variant):12.2f}" for n in (24000, 48000) for variant in GRID)
          + f" {PREDICTED[snr]:9.2f}")

In [ ]:
snrs = list(EPS)
fig, ax = plt.subplots(figsize=(8, 4.3), constrained_layout=True)
ax.plot(snrs, [ratio(24000, s, "sKF") for s in snrs], "o-", color=COLORS[0], lw=2, ms=8,
        label="sKF-L, variance recursion (rerun of notebook 03)")
ax.plot(snrs, [ratio(24000, s, "fKF") for s in snrs], "s-", color=COLORS[1], lw=2, ms=8,
        label=r"fixed variance, $\tilde v_t$ frozen")
ax.plot(snrs, [PREDICTED[s] for s in snrs], "^--", color="k", lw=1.5, ms=8,
        label="analysis, section 6 of the draft (white input, M = 16)")
ax.axhline(1, color="k", lw=0.8)
ax.set(xticks=snrs, xlabel="SNR [dB]", ylabel="steps, exact / minorized",
       title=f"Equal floor ({TARGET_DB:.0f} dB), N = 24000, R = {R_CHECK}: above 1, the minorized filter is faster")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()

### Findings

* **The variance recursion does not explain the gap.** With $\tilde v_t$ frozen, the exact filter still needs 1.54
  times the minorized filter's steps at 0 dB (1.55 at $N = 48000$), against 1.63 (1.56) with the recursion. At every
  SNR and both lengths the two versions differ by at most 0.09, with no consistent sign, and the fixed-variance
  ratios stay well above the analysis (1.13 at 0 dB).
* **That difference is within the protocol's noise.** Changing only $N$ and $R$ moves the same filter's ratio by up
  to 0.07: sKF at 0 dB, 1.63 at $N = 24000$ and 1.56 at $N = 48000$; at 5 dB, 1.27 here and 1.34 in notebook 03's
  long run.
* **The rerun reproduces notebook 03.** At $N = 24000$ the sKF rows land on its $\varepsilon$ (within 0.3 %, its
  values are rounded) and on its steps at 0, 10 and 15 dB (6213 / 10151, 3418 / 3812, 3081 / 3180).
* **The floors had settled.** Every selected point is within 0.21 dB of the target, and doubling $N$ leaves the
  conclusion unchanged.
* **What is left.** The fixed-variance filters here still differ from the analysis in the input (AR(1) with
  $a = -0.9$, against white), in $M$ (128 against 16), and in how the floor is read (last quarter of a finite run,
  against the steady state of the recursion). Next, one change at a time: white input at $M = 128$, then $M = 16$.
  If the last one lands near 1.13, simulation and analysis agree and the extra gap comes from the input or $M$.